# 04 — Fusion INSEE (année n-1) × Résultats municipaux + première analyse

**Objectif** : associer à chaque commune, pour chaque élection municipale, les
variables socio-démographiques INSEE **de l'année précédant le scrutin**, puis
regarder si des variables INSEE sont liées au résultat du vote (bloc politique,
participation).

**Table année élection → année INSEE utilisée :**

| Élection | Année INSEE |
|---|---|
| 2008 | 2007 |
| 2014 | 2013 |
| 2020 | 2019 |
| 2026 | 2019 *(pas de recensement 2025 disponible dans nos données — à date de rédaction, le dernier millésime exploité est 2019 ; si tu récupères un recensement plus récent, change juste `ANNEE_ELECTION_TO_INSEE`)* |

L'élection **2001 n'a pas d'année INSEE antérieure disponible** dans ce jeu de
données (le premier recensement qu'on a est 2007, donc *après* 2001) — elle est
exclue de cette fusion.

⚠️ **Fusions de communes** : entre 2007 et 2026, des centaines de communes ont
fusionné ("communes nouvelles"), ce qui change leur code INSEE dans le temps.
Une jointure directe sur `code_insee` sans correction ferait perdre ou mal
apparier ces communes. On corrige ça avec la **table de passage officielle de
l'INSEE**.


## 1. Table de passage des communes (fusions) — à télécharger une fois

Fichier officiel INSEE (page *Tables de passage des communes*) :

**Table de passage géographie 2003 → géographie 2026**
https://www.insee.fr/fr/statistiques/fichier/7671867/table_passage_geo2003_geo2026.zip

Cette table donne, pour **toute commune ayant existé depuis 2003**, son code
correspondant dans la géographie communale actuelle (2026) — donc y compris les
communes de 2007/2013/2019 qui ont depuis fusionné.

**Action à faire une seule fois** : télécharge le zip, dézippe-le, et place le
CSV qu'il contient dans `data/raw/table_passage_geo2003_geo2026.csv`
(le sandbox où je travaille n'a pas accès à insee.fr, donc je ne peux pas le
télécharger à ta place).

Le format exact des colonnes a un peu varié selon les millésimes (voir
avertissement INSEE : "les fichiers ont sensiblement évolué à partir du
millésime 2019"). La cellule suivante affiche les colonnes pour que tu
vérifies/adaptes `COL_CODE_AVANT` / `COL_CODE_APRES` si besoin.


In [1]:
import os
import sys
import gc

import pandas as pd
import numpy as np

sys.path.append(os.path.abspath('../..'))
pd.set_option('display.max_columns', 60)

RAW_DIR = os.path.abspath('../../data/raw')
PROC_DIR = os.path.abspath('../../data/processed')

table_passage_path = os.path.join(RAW_DIR, "table_passage_geo2003_geo2026.xlsx")
df_passage_raw = pd.read_excel(table_passage_path, skiprows=5)
print(df_passage_raw.shape)
df_passage_raw.head()


(36743, 3)


,CODGEO_INI,CODGEO_2026,LIBGEO_2026
0,01001,01001,L'Abergement-Clémenciat
1,01002,01002,L'Abergement-de-Varey
2,01004,01004,Ambérieu-en-Bugey
3,01005,01005,Ambérieux-en-Dombes
4,01006,01006,Ambléon


In [2]:
# À AJUSTER selon ce que tu vois dans les colonnes ci-dessus.
# Noms usuels dans les tables de passage INSEE post-2019 : CODGEO_INI / CODGEO_2026
# (ou COD_...  /  LIBGEO_...). On essaie de les détecter automatiquement.

def guess_col(df, *keywords):
    for c in df.columns:
        cl = c.lower()
        if all(k in cl for k in keywords):
            return c
    return None

COL_CODE_AVANT = guess_col(df_passage_raw, "codgeo") or guess_col(df_passage_raw, "cod")
COL_CODE_APRES = None
for c in df_passage_raw.columns:
    if c != COL_CODE_AVANT and "2026" in c.lower() and "cod" in c.lower():
        COL_CODE_APRES = c
        break

print("Colonne code avant  :", COL_CODE_AVANT)
print("Colonne code après  :", COL_CODE_APRES)
assert COL_CODE_AVANT and COL_CODE_APRES, (
    "Détection automatique échouée : regarde df_passage_raw.columns et fixe "
    "COL_CODE_AVANT / COL_CODE_APRES à la main."
)


Colonne code avant  : CODGEO_INI
Colonne code après  : CODGEO_2026


In [3]:
# Dictionnaire de correspondance : n'importe quel code_insee historique (depuis 2003)
# -> code_insee dans la géographie 2026 (référentiel commun choisi pour toutes les
# fusions de ce projet).
code_to_2026 = (
    df_passage_raw[[COL_CODE_AVANT, COL_CODE_APRES]]
    .dropna()
    .drop_duplicates(subset=COL_CODE_AVANT)
    .set_index(COL_CODE_AVANT)[COL_CODE_APRES]
    .to_dict()
)
print(f"{len(code_to_2026):,} correspondances de code commune chargées.")

def harmonize_code(series: pd.Series) -> pd.Series:
    """Convertit une série de codes commune vers le référentiel géo 2026.
    Les codes déjà à jour ou absents de la table (DOM/TOM, cas particuliers)
    sont laissés inchangés plutôt que mis à NaN."""
    s = series.astype(str).str.strip().str.zfill(5)
    return s.map(code_to_2026).fillna(s)


36,714 correspondances de code commune chargées.


## 2. Chargement des INSEE fusionnés (notebook 03) et des résultats municipaux

`Analysis_Municipale.csv` vient de `01_EDA_Municipales_Voix.ipynb` (déjà nettoyé
du panachage, avec `population`, `strate`, `taux_participation`, etc.).


In [4]:
insee_frames = {}
for year in (2007, 2013, 2019):
    path = os.path.join(PROC_DIR, f"INSEE_Fusion_{year}.csv")
    df_y = pd.read_csv(path, sep=";", low_memory=False)
    df_y['code_insee_2026'] = harmonize_code(df_y['code_insee'])
    insee_frames[year] = df_y
    print(year, df_y.shape)


2007 (36323, 46)
2013 (36641, 52)
2019 (34938, 52)


In [5]:
df_municipales = pd.read_csv(
    os.path.join(PROC_DIR, "Analysis_Geo.csv"), low_memory=False
)
df_municipales

,Annee,Tour,Code département,Code commune,Libellé commune,Code BV,Inscrits,Votants,Abstentions,Exprimés,Code Nuance,Voix,Nuance,Libellé,code_insee,geo_nom,population,surface_km2,longitude,latitude,ID Bureau de vote,Vote Blanc,strate,bureau_anormal,taux_participation,taux_abstention,taux_vote_blanc_nul,nb_sieges_estime,pct_voix_exprimes_brut,pct_voix_exprimes_normalise,bloc_politique
0,2008,1,1,4,Ambérieu-en-Bugey,0001,1100,622,478,589,LUG,172.0,LUG,Liste Union de la Gauche,1004,Ambérieu-en-Bugey,15934.0,2454.8,5.3706,45.9575,1,33,>= 1000 hab,False,0.565455,0.434545,0.053055,1.0,0.292020,0.292020,Gauche
1,2008,1,1,4,Ambérieu-en-Bugey,0001,1100,622,478,589,LGC,111.0,LGC,Liste gauche-centristes,1004,Ambérieu-en-Bugey,15934.0,2454.8,5.3706,45.9575,1,33,>= 1000 hab,False,0.565455,0.434545,0.053055,1.0,0.188455,0.188455,Gauche
2,2008,1,1,4,Ambérieu-en-Bugey,0001,1100,622,478,589,LDVD,214.0,LDVD,Liste divers droite,1004,Ambérieu-en-Bugey,15934.0,2454.8,5.3706,45.9575,1,33,>= 1000 hab,False,0.565455,0.434545,0.053055,1.0,0.363328,0.363328,Droite
3,2008,1,1,4,Ambérieu-en-Bugey,0001,1100,622,478,589,LDVD,92.0,LDVD,Liste divers droite,1004,Ambérieu-en-Bugey,15934.0,2454.8,5.3706,45.9575,1,33,>= 1000 hab,False,0.565455,0.434545,0.053055,1.0,0.156197,0.156197,Droite
4,2008,1,1,4,Ambérieu-en-Bugey,0002,935,497,438,477,LUG,148.0,LUG,Liste Union de la Gauche,1004,Ambérieu-en-Bugey,15934.0,2454.8,5.3706,45.9575,2,20,>= 1000 hab,False,0.531551,0.468449,0.040241,1.0,0.310273,0.310273,Gauche
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1584246,2026,2,95,95680,Villiers-le-Bel,0013,946,508,438,505,LFI,260.0,LFI,La France insoumise,9595680,NaN,946.0,NaN,NaN,NaN,13,3,< 1000 hab,False,0.536998,0.463002,0.005906,NaN,0.514851,0.000000,Gauche
1584247,2026,2,95,95680,Villiers-le-Bel,0014,993,487,506,476,LSOC,258.0,LSOC,Liste du Parti socialiste,9595680,NaN,993.0,NaN,NaN,NaN,14,11,< 1000 hab,False,0.490433,0.509567,0.022587,NaN,0.542017,0.000000,Gauche
1584248,2026,2,95,95680,Villiers-le-Bel,0014,993,487,506,476,LFI,218.0,LFI,La France insoumise,9595680,NaN,993.0,NaN,NaN,NaN,14,11,< 1000 hab,False,0.490433,0.509567,0.022587,NaN,0.457983,0.000000,Gauche
1584249,2026,2,95,95680,Villiers-le-Bel,0015,1086,577,509,564,LSOC,359.0,LSOC,Liste du Parti socialiste,9595680,NaN,1086.0,NaN,NaN,NaN,15,13,>= 1000 hab,False,0.531308,0.468692,0.022530,1.0,0.636525,0.636525,Gauche


In [6]:
df_municipales['code_insee_2026'] = harmonize_code(df_municipales['code_insee'])
print(df_municipales.shape)
print(sorted(df_municipales['Annee'].unique()))


(1584251, 32)
[np.int64(2008), np.int64(2014), np.int64(2020), np.int64(2026)]


## 3. Fusion élection × INSEE (année n-1)

On fusionne, pour chaque élection, avec la table INSEE correspondante — sur le
code commune **harmonisé** (`code_insee_2026`), pas sur le code brut de l'époque.


In [7]:
ANNEE_ELECTION_TO_INSEE = {
    2008: 2007,
    2014: 2013,
    2020: 2019,
    2026: 2019,  # dernier recensement disponible dans nos données à ce stade
}

merged_parts = []
for annee_election, annee_insee in ANNEE_ELECTION_TO_INSEE.items():
    sous_election = df_municipales[df_municipales['Annee'] == annee_election]
    if sous_election.empty:
        print(f"Aucune ligne pour l'élection {annee_election}, on passe.")
        continue
    df_insee_annee = insee_frames[annee_insee]

    fusion = sous_election.merge(
        df_insee_annee.drop(columns=['code_insee']),  # évite le doublon avec la clé harmonisée
        on='code_insee_2026',
        how='left',
        suffixes=('', f'__insee{annee_insee}'),
    )
    taux_non_apparie = fusion['annee_insee'].isna().mean() * 100
    print(
        f"Élection {annee_election} (INSEE {annee_insee}) : {len(sous_election):,} lignes, "
        f"{taux_non_apparie:.1f}% sans correspondance INSEE"
    )
    merged_parts.append(fusion)

df_final = pd.concat(merged_parts, ignore_index=True)
print("\nTaille finale :", df_final.shape)


Élection 2008 (INSEE 2007) : 128,084 lignes, 100.0% sans correspondance INSEE
Élection 2014 (INSEE 2013) : 666,727 lignes, 0.0% sans correspondance INSEE
Élection 2020 (INSEE 2019) : 560,581 lignes, 0.0% sans correspondance INSEE
Élection 2026 (INSEE 2019) : 228,859 lignes, 100.0% sans correspondance INSEE

Taille finale : (1729300, 91)


### Vérification des communes non appariées

Avant de continuer, on regarde *qui* n'a pas été apparié : si c'est concentré
sur quelques départements/années précis, c'est probablement un souci de format
de code plutôt qu'une vraie absence de donnée INSEE.


In [8]:
non_apparie = df_final[df_final['annee_insee'].isna()]
print(f"{len(non_apparie):,} lignes sans correspondance INSEE ({len(non_apparie) / len(df_final) * 100:.1f}%)")
display(
    non_apparie.groupby('Annee')['code_insee_2026'].nunique()
    .rename('communes_non_appariees')
)
non_apparie[['Annee', 'code_insee', 'code_insee_2026', 'Libellé commune']].drop_duplicates().head(20)


356,943 lignes sans correspondance INSEE (20.6%)


Annee
2008     2439
2026    34311
Name: communes_non_appariees, dtype: int64

,Annee,code_insee,code_insee_2026,Libellé commune
0,2008,1004,01004,Ambérieu-en-Bugey
32,2008,1014,01014,Arbent
36,2008,1033,01033,Bellegarde-sur-Valserine
50,2008,1034,01034,Belley
65,2008,1043,01043,Beynost
74,2008,1053,01053,Bourg-en-Bresse
150,2008,1093,01093,Châtillon-sur-Chalaronne
159,2008,1142,01142,Dagneux
163,2008,1143,01143,Divonne-les-Bains
168,2008,1160,01160,Ferney-Voltaire


In [9]:
df_final.to_csv(os.path.join(PROC_DIR, "Municipales_INSEE_Merged.csv"), index=False, sep=";")
print("Exporté -> data/processed/Municipales_INSEE_Merged.csv", df_final.shape)

del insee_frames, merged_parts
gc.collect()


Exporté -> data/processed/Municipales_INSEE_Merged.csv (1729300, 91)


0

## 4. Première analyse : variables INSEE vs vote

On repart du calcul du **bloc politique majoritaire** par commune × année
(logique déjà posée dans `02_EDA_Municipales_Geo.ipynb`), pour croiser avec les
variables socio-démographiques fraîchement rattachées.

⚠️ Les noms de colonnes ci-dessous (`taux_chomage`, `pct_immigres`, ...) sont
**des exemples à remplacer** par les vraies colonnes de tes tables fusionnées
(`__emploi`, `__immigration`, etc. selon les suffixes posés dans le notebook 03)
— lance `df_final.filter(like='emploi').columns` pour les retrouver.


In [10]:
# Repérer les colonnes par thème grâce aux suffixes posés dans le notebook 03
for theme in ['emploi', 'diplome', 'immigration', 'nationalite', 'logement', 'menage', 'famille']:
    cols = [c for c in df_final.columns if theme in c.lower()]
    print(f"{theme:12s}: {len(cols)} colonnes -> {cols[:5]}{'...' if len(cols) > 5 else ''}")


emploi      : 13 colonnes -> ['SEXE__emploi1', 'CS3_31__emploi1', 'NA5__emploi1', 'Nombre__emploi1', 'SEXE__emploi2']...
diplome     : 5 colonnes -> ['SEXE__diplome', 'AGEQ65__diplome', 'DIPL__diplome', 'Nombre__diplome', 'LIBGEO__diplome']
immigration : 7 colonnes -> ['SEXE__immigration', 'AGE4__immigration', 'TACTR__immigration', 'IMMI__immigration', 'Nombre__immigration']...
nationalite : 10 colonnes -> ['SEXE__nationalite', 'INATC__nationalite', 'AGE4__nationalite', 'Nombre__nationalite', 'SEXE__nationalite2']...
logement    : 5 colonnes -> ['TYPLR__logement', 'CS1_8__logement', 'STOCD__logement', 'Nombre__logement', 'LIBGEO__logement']
menage      : 4 colonnes -> ['CS2_24__menage', 'NPERC__menage', 'Nombre__menage', 'LIBGEO__menage']
famille     : 4 colonnes -> ['CS2_24__famille', 'NBENFFR__famille', 'Nombre__famille', 'LIBGEO__famille']


In [11]:
from scipy.stats import spearmanr

# Reprend le bloc politique majoritaire par commune x année (cf. notebook 02)
mask_classe = ~df_final['bloc_politique'].isin(['Non renseigné', 'Non classé', 'Non Communiqué'])
score_bloc = (
    df_final[mask_classe]
    .groupby(['Annee', 'code_insee_2026', 'bloc_politique'])['pct_voix_exprimes_normalise']
    .sum()
    .reset_index()
)
bloc_majoritaire = (
    score_bloc.sort_values('pct_voix_exprimes_normalise', ascending=False)
    .drop_duplicates(subset=['Annee', 'code_insee_2026'])
    .rename(columns={'bloc_politique': 'bloc_majoritaire'})
)

# Une ligne par commune x année, avec ses variables INSEE (on ne garde qu'une ligne
# par commune x année pour éviter de dupliquer les variables INSEE par candidat)
commune_annee = df_final.drop_duplicates(subset=['Annee', 'code_insee_2026'])
analyse = bloc_majoritaire.merge(
    commune_annee, on=['Annee', 'code_insee_2026'], how='left', suffixes=('', '_dup')
)
analyse

,Annee,code_insee_2026,bloc_majoritaire,pct_voix_exprimes_normalise,Tour,Code département,Code commune,Libellé commune,Code BV,Inscrits,Votants,Abstentions,Exprimés,Code Nuance,Voix,Nuance,Libellé,code_insee,geo_nom,population,surface_km2,longitude,latitude,ID Bureau de vote,Vote Blanc,strate,bureau_anormal,taux_participation,taux_abstention,taux_vote_blanc_nul,...,Nombre__nationalite2,SEXE__immigration,AGE4__immigration,TACTR__immigration,IMMI__immigration,Nombre__immigration,SEXE__emploi1,CS3_31__emploi1,NA5__emploi1,Nombre__emploi1,SEXE__emploi2,CS3_31__emploi2,NA5__emploi2,Nombre__emploi2,annee_insee,LIBGEO__famille,LIBGEO__menage,LIBGEO__diplome,LIBGEO__logement,LIBGEO__population,LIBGEO__population2,LIBGEO__nationalite,LIBGEO__nationalite2,LIBGEO__immigration,AGE4_A__immigration,LIBGEO__emploi,CS3_29__emploi,NA5__emploi,SEXE__emploi,Nombre__emploi
0,2014,75056,Gauche,585945.000000,1,75,56,Paris,0101,1,1,0,1,LPG,14.0,LPG,Liste du Parti de Gauche,75056,Paris,2103778.0,10536.03,2.3470,48.8589,1,0,>= 1000 hab,False,1.000000,0.000000,0.000000,...,168.0,Hommes,NaN,Actifs ayant un emploi,Immigrés,4801.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2013.0,Paris,Paris,Paris,Paris,Paris,Paris,Paris,Paris,Paris,15 à 24 ans,Paris,Agriculteurs exploitants,"Agriculture, sylviculture et pêche",Hommes,129.0
1,2014,13055,Droite,162359.000000,1,13,55,Marseille,0101,2,1,1,1,LUG,126.0,LUG,Liste Union de la Gauche,13055,Marseille,886040.0,23793.28,5.3806,43.2803,1,0,>= 1000 hab,False,0.500000,0.500000,0.000000,...,134.0,Hommes,NaN,Actifs ayant un emploi,Immigrés,1133.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2013.0,Marseille,Marseille,Marseille,Marseille,Marseille,Marseille,Marseille,Marseille,Marseille,15 à 24 ans,Marseille,Agriculteurs exploitants,"Agriculture, sylviculture et pêche",Hommes,98.0
2,2014,69123,Gauche,73861.000000,1,69,123,Lyon,0101,2,1,1,1,LEXG,4.0,LEXG,Liste d'extrême-gauche,69123,Lyon,519127.0,4797.43,4.8351,45.7580,1,0,>= 1000 hab,False,0.500000,0.500000,0.000000,...,70.0,Hommes,NaN,Actifs ayant un emploi,Immigrés,998.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2013.0,Lyon,Lyon,Lyon,Lyon,Lyon,Lyon,Lyon,Lyon,Lyon,15 à 24 ans,Lyon,Agriculteurs exploitants,"Agriculture, sylviculture et pêche",Hommes,30.0
3,2014,31555,Droite,59064.333333,1,31,555,Toulouse,0001,4,2,2,2,LUD,215.0,LUD,Liste Union de la Droite,31555,Toulouse,514819.0,11805.53,1.4328,43.6007,1,0,>= 1000 hab,False,0.500000,0.500000,0.000000,...,87.0,Hommes,NaN,Actifs ayant un emploi,Immigrés,1148.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2013.0,Toulouse,Toulouse,Toulouse,Toulouse,Toulouse,Toulouse,Toulouse,Toulouse,Toulouse,15 à 24 ans,Toulouse,Agriculteurs exploitants,"Agriculture, sylviculture et pêche",Hommes,106.0
4,2014,06088,Droite,57189.666667,1,6,88,Nice,0101,3,1,2,1,LDVD,11.0,LDVD,Liste divers droite,6088,Nice,357737.0,7380.49,7.2528,43.7032,1,0,>= 1000 hab,False,0.333333,0.666667,0.000000,...,111.0,Hommes,NaN,Actifs ayant un emploi,Immigrés,654.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2013.0,Nice,Nice,Nice,Nice,Nice,Nice,Nice,Nice,Nice,15 à 24 ans,Nice,Agriculteurs exploitants,"Agriculture, sylviculture et pêche",Hommes,111.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17890,2026,4040036,Gauche,0.000000,1,40,40036,Bénesse-Maremne,0001,688,284,404,245,LDVG,245.0,LDVG,Liste divers gauche,4040036,NaN,688.0,NaN,NaN,NaN,1,39,< 1000 hab,False,0.412791,0.587209,0.137324,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17891,2026,404019,Gauche,0.000000,1,4,4019,Barcelonnette,0001,998,591,407,565,LDVD,65.0,LDVD,Liste divers droite,404019,NaN,998.0,NaN,NaN,NaN,1,26,< 1000 hab,False,0.592184,0.407816,0.043993,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17892,

In [12]:
# Exemple de corrélation Spearman entre une variable INSEE (à adapter, cf. cellule
# précédente pour les vrais noms) et le score du bloc majoritaire
colonnes_insee_a_tester = [
    c for c in analyse.columns
    if any(theme in c.lower() for theme in ['emploi', 'diplome', 'immigration', 'nationalite'])
    and pd.api.types.is_numeric_dtype(pd.to_numeric(analyse[c], errors='coerce'))
]
colonnes_insee_a_tester

['SEXE__diplome',
 'AGEQ65__diplome',
 'DIPL__diplome',
 'Nombre__diplome',
 'SEXE__nationalite',
 'INATC__nationalite',
 'AGE4__nationalite',
 'Nombre__nationalite',
 'SEXE__nationalite2',
 'INATC__nationalite2',
 'CS1_8__nationalite2',
 'Nombre__nationalite2',
 'SEXE__immigration',
 'AGE4__immigration',
 'TACTR__immigration',
 'IMMI__immigration',
 'Nombre__immigration',
 'SEXE__emploi1',
 'CS3_31__emploi1',
 'NA5__emploi1',
 'Nombre__emploi1',
 'SEXE__emploi2',
 'CS3_31__emploi2',
 'NA5__emploi2',
 'Nombre__emploi2',
 'LIBGEO__diplome',
 'LIBGEO__nationalite',
 'LIBGEO__nationalite2',
 'LIBGEO__immigration',
 'AGE4_A__immigration',
 'LIBGEO__emploi',
 'CS3_29__emploi',
 'NA5__emploi',
 'SEXE__emploi',
 'Nombre__emploi']

In [13]:
resultats = []
for col in colonnes_insee_a_tester:
    x = pd.to_numeric(analyse[col], errors='coerce')
    y = pd.to_numeric(analyse['pct_voix_exprimes_normalise'], errors='coerce')
    valid = pd.concat([x, y], axis=1).dropna()
    if len(valid) < 30 or valid[col].nunique() < 2:
        continue
    try:
        rho, p = spearmanr(valid.iloc[:, 0].to_numpy(), valid.iloc[:, 1].to_numpy())
    except Exception as e:
        print(f"⚠️ échec sur '{col}' : {e}")
        continue
    resultats.append({'variable_insee': col, 'rho_spearman': rho, 'p_value': p})

corr_insee = pd.DataFrame(resultats).sort_values('rho_spearman', key=abs, ascending=False)
corr_insee

,variable_insee,rho_spearman,p_value
1,Nombre__nationalite,0.749825,0.000000
0,Nombre__diplome,0.642405,0.000000
3,Nombre__immigration,0.536672,0.000000
2,Nombre__nationalite2,0.026799,0.002944
4,Nombre__emploi,0.020805,0.020986


## 5. Synthèse et suite

À compléter une fois exécuté sur les vraies données :
- **Taux d'appariement INSEE** (§3) : est-il homogène entre années/départements,
  ou y a-t-il un biais lié aux fusions de communes non couvertes par la table
  de passage ?
- **Corrélations INSEE ↔ vote** (§4) : quelles variables socio-démographiques
  se démarquent (|rho| le plus élevé, p < 0.05) ? Sont-elles cohérentes avec la
  littérature électorale (chômage, diplôme, part d'immigrés, structure de
  l'habitat) ?
- **Prochaine étape modélisation** : `Municipales_INSEE_Merged.csv` est la base
  pour construire les features du modèle de prédiction (page `5_Machine_Learning.py`
  / `6_Prediction.py` de l'app).
